# E2 — Exportar os resultados no formato do trabalho da IC

Converte `results/e2/<CONFIG>/` (saída da campanha) para a **mesma estrutura de pastas e arquivos** que o
`TrainingMGGP_LOOP.ipynb` da IC produzia, de modo que `BOXPLOT.ipynb`, `mineracao/` e
`Relatorio_resultados/` possam ler os resultados novos sem alteração.

**Somente leitura na fonte.** Este notebook nunca escreve em `results/e2/<CONFIG>/` e nunca roda a
campanha — pode ser executado com ela em andamento (ele exporta o que já foi aprovado até agora).
Tudo é gerado em `results/e2/<CONFIG>_formato_IC/`, e pode ser apagado e refeito à vontade.

| IC (`RESULTADOS-N/Treinamento_ID/`) | aqui (`RESULTADOS-E2/Treinamento_N/`) |
|---|---|
| `TRAINING/modelo_rmse_N.pkl` | idem (cópia dos modelos aprovados) |
| `modelo_rmse_N.png` — 2 colunas (Vx, Vy) por pista | idem, redesenhado das predições salvas (`.npz`), sem refazer Free-Run; agora com 5 linhas (Wang 3.2 entrou) |
| `relatorio_validacao_rmse.csv` — `;` e vírgula decimal, colunas `Modelo;Pista;RMSE_Vx;RMSE_Vy;RMSE_Médio` | idem, byte a byte no formato; pista que divergiu = `999.9`, como na IC |
| `parametros_utilizados.csv` | idem, mesma ordem de chaves |
| `checklist_local.xlsx` | idem |
| — | `beta_validacao.csv` (novo): RMSE do β por modelo e pista |
| — | `tentativas.csv` (novo): ledger com seed, status e tempo de cada tentativa |

Como uma campanha tem **um** braço por configuração, cada braço vira uma pasta `Treinamento_N`:
`Treinamento_1` = 5×2 (vx, vy), `Treinamento_2` = 5×3 (vx, vy, β).

In [ ]:
# Configuração — normalmente só o CONFIG muda.
CONFIG  = "t48"
FONTE   = None     # None = <repo>/results/e2/<CONFIG>          (somente leitura)
DESTINO = None     # None = <repo>/results/e2/<CONFIG>_formato_IC
BRACOS  = {"5x2": "Treinamento_1", "5x3": "Treinamento_2"}   # braço -> pasta, no estilo Treinamento_N da IC

In [ ]:
import shutil
import time
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt


def _achar_raiz(inicio):
    """Raiz do repositório = primeira pasta (subindo) que contém src/mggp.py."""
    for d in [inicio, *inicio.parents]:
        if (d / "src" / "mggp.py").exists():
            return d
    raise RuntimeError(f"não achei a raiz do repositório a partir de {inicio}")


ROOT = _achar_raiz(Path.cwd().resolve())
FONTE_DIR = Path(FONTE) if FONTE else ROOT / "results" / "e2" / CONFIG
DESTINO_DIR = Path(DESTINO) if DESTINO else ROOT / "results" / "e2" / f"{CONFIG}_formato_IC"
BASE_DIR = DESTINO_DIR / "RESULTADOS-E2"

# Nomes e ordem das pistas, idênticos aos do relatório da IC (+ Wang 3.2, que a IC não usou).
PISTAS = [
    ("Wang 2.1 (Treino)", "train"),
    ("Wang 2.2 (Validação)", "val"),
    ("Wang 3.1 (Teste A)", "testA"),
    ("Wang 8.1 (Teste B)", "testB"),
    ("Wang 3.2 (Teste C)", "testC"),
]
# Ordem de chaves do parametros_utilizados.csv da IC.
ORDEM_PARAMS = ["nDelays", "generations", "populationSize", "evaluationMode", "k", "evaluationType",
                "evaluationTypeTest", "nTerms", "maxHeight", "mutationRate", "crossoverRate",
                "elitePercentage", "mode", "froe_mode", "operators"]
FALHA = 999.9   # convenção da IC para uma pista em que o Free-Run falhou

if not FONTE_DIR.exists():
    raise FileNotFoundError(f"não existe {FONTE_DIR} — rode a campanha (ou copie results/e2/ do outro PC) primeiro")
print(f"fonte   : {FONTE_DIR}  (somente leitura)")
print(f"destino : {BASE_DIR}")


def ler_csv(caminho, tentativas=3):
    """Lê um CSV; se a campanha estiver gravando o arquivo agora, tenta de novo."""
    for i in range(tentativas):
        try:
            return pd.read_csv(caminho)
        except (pd.errors.ParserError, pd.errors.EmptyDataError):
            if i == tentativas - 1:
                raise
            time.sleep(1)


def numero(nome):
    """'modelo_rmse_12' -> 12"""
    return int(nome.rsplit("_", 1)[1])

In [ ]:
def desenhar_figura_IC(nome, linhas_por_pista, npz, destino):
    """Mesma figura do notebook da IC: Vx (azul tracejado) e Vy (vermelho tracejado) contra o real (preto)."""
    n = len(PISTAS)
    fig, axes = plt.subplots(n, 2, figsize=(18, 5 * n))
    fig.suptitle(f"Validação Completa (Dados Originais): {nome}.pkl", fontsize=16, y=1.01)
    for i, (pista, papel) in enumerate(PISTAS):
        r = linhas_por_pista.get(pista)
        yd, yp = npz[f"{papel}_yd"], npz[f"{papel}_yp"]
        ok = r is not None and bool(r["ok"]) and bool(np.all(np.isfinite(yp)))
        for j, (rotulo, cor, chave) in enumerate([("Vx", "blue", "RMSE_Vx"), ("Vy", "red", "RMSE_Vy")]):
            ax = axes[i, j]
            ax.plot(yd[:, j], color="black", label="Real")
            if ok:
                ax.plot(yp[:, j], color=cor, linestyle="--", label="Previsto")
                ax.set_title(f"{pista} - {rotulo} | RMSE: {float(r[chave]):.4f}")
            else:
                ax.set_title(f"{pista} - {rotulo} | Free-Run divergiu")
            ax.grid(True, linestyle=":", alpha=0.6)
    plt.tight_layout()
    fig.savefig(destino, bbox_inches="tight")
    plt.close(fig)


def exportar_braco(arm, nome_pasta):
    src = FONTE_DIR / arm
    if not (src / "relatorio_validacao_rmse.csv").exists():
        print(f"[{arm}] ainda sem relatorio_validacao_rmse.csv — nada a exportar")
        return None
    rep = ler_csv(src / "relatorio_validacao_rmse.csv")
    out = BASE_DIR / nome_pasta
    (out / "TRAINING").mkdir(parents=True, exist_ok=True)

    modelos = sorted(rep["Modelo"].unique(), key=numero)
    tem_saida3 = "RMSE_beta_out_deg" in rep.columns
    linhas, beta = [], []
    for m in modelos:
        sub = rep[rep["Modelo"] == m].set_index("Pista")
        for pista, _ in PISTAS:
            if pista not in sub.index:
                continue
            r = sub.loc[pista]
            ok = bool(r["ok"])
            vx, vy, med = ((round(float(r[c]), 4) for c in ("RMSE_Vx", "RMSE_Vy", "RMSE_Medio_VxVy"))
                           if ok else (FALHA, FALHA, FALHA))
            linhas.append({"Modelo": f"{m}.pkl", "Pista": pista, "RMSE_Vx": vx, "RMSE_Vy": vy, "RMSE_Médio": med})
            b = {"Modelo": f"{m}.pkl", "Pista": pista,
                 "RMSE_Beta_atan2_deg": round(float(r["RMSE_beta_atan2_deg"]), 4) if ok else FALHA}
            if tem_saida3:
                b["RMSE_Beta_saida3_deg"] = round(float(r["RMSE_beta_out_deg"]), 4) if ok else FALHA
            beta.append(b)

    # --- relatórios no formato da IC: separador ';' e vírgula decimal
    pd.DataFrame(linhas).to_csv(out / "relatorio_validacao_rmse.csv", index=False, sep=";", decimal=",")
    pd.DataFrame(beta).to_csv(out / "beta_validacao.csv", index=False, sep=";", decimal=",")

    # --- modelos e figuras
    faltando = []
    for m in modelos:
        pkl = src / "models" / f"{m}.pkl"
        if pkl.exists():
            shutil.copy2(pkl, out / "TRAINING" / f"{m}.pkl")
        else:
            faltando.append(f"{m}.pkl")
        npz = src / "predictions" / f"{m}.npz"
        if npz.exists():
            with np.load(npz) as z:
                por_pista = {p: r for p, r in rep[rep["Modelo"] == m].set_index("Pista").iterrows()}
                desenhar_figura_IC(m, por_pista, z, out / f"{m}.png")
        else:
            faltando.append(f"{m}.npz")

    # --- parametros_utilizados.csv na ordem de chaves da IC
    pj = FONTE_DIR / "params.json"
    if pj.exists():
        import json
        params = json.loads(pj.read_text(encoding="utf-8"))["params"]
        itens = [(k, params[k]) for k in ORDEM_PARAMS if k in params]
        pd.DataFrame(itens, columns=["Parametro", "Valor"]).to_csv(out / "parametros_utilizados.csv", index=False)

    # --- checklist_local.xlsx (todos os modelos exportados já passaram por treino e validação)
    n = len(modelos)
    pd.DataFrame({"Etapa": [f"Treino_{i:02d}" for i in range(1, n + 1)] + [f"Validacao_{i:02d}" for i in range(1, n + 1)],
                  "Status": ["Concluído"] * (2 * n)}).to_excel(out / "checklist_local.xlsx", index=False)

    # --- ledger de tentativas (novo)
    if (src / "attempts.csv").exists():
        shutil.copy2(src / "attempts.csv", out / "tentativas.csv")

    print(f"[{arm}] -> {out.name}: {n} modelo(s), {len(linhas)} linhas no relatório"
          + (f" · FALTANDO: {faltando}" if faltando else ""))
    return {"pasta": out, "modelos": n, "linhas": len(linhas), "faltando": faltando}

In [ ]:
BASE_DIR.mkdir(parents=True, exist_ok=True)
resumo = {arm: exportar_braco(arm, pasta) for arm, pasta in BRACOS.items()}

(BASE_DIR / "LEIAME.txt").write_text(
    "Exportado de results/e2/%s no formato do trabalho da IC (RESULTADOS-N/Treinamento_ID).\n\n"
    "  Treinamento_1 = braco 5x2 (saidas: vx, vy)\n"
    "  Treinamento_2 = braco 5x3 (saidas: vx, vy, beta)\n\n"
    "relatorio_validacao_rmse.csv : sep=';' decimal=',' (igual a IC). 999.9 = Free-Run divergiu na pista.\n"
    "                               Inclui Wang 3.2 (Teste C), que a IC nao usou.\n"
    "beta_validacao.csv           : RMSE do beta em graus (atan2 das velocidades preditas; e a saida 3 no 5x3).\n"
    "tentativas.csv               : todas as tentativas (seed, status, RMSE do gate, tempo).\n" % CONFIG,
    encoding="utf-8")
print("\npronto:", BASE_DIR)

## Conferência

Lê os arquivos gerados **do mesmo jeito que o `BOXPLOT.ipynb` e o `mineracao/` leem** (`sep=';'`, `decimal=','`)
e mostra o começo de cada um.

In [ ]:
for arm, pasta in BRACOS.items():
    p = BASE_DIR / pasta
    if not (p / "relatorio_validacao_rmse.csv").exists():
        continue
    df = pd.read_csv(p / "relatorio_validacao_rmse.csv", sep=";", decimal=",")
    assert list(df.columns) == ["Modelo", "Pista", "RMSE_Vx", "RMSE_Vy", "RMSE_Médio"], df.columns
    assert df[["RMSE_Vx", "RMSE_Vy", "RMSE_Médio"]].dtypes.eq(float).all(), "decimal não foi lido como número"
    print(f"=== {arm} · {pasta} — {df['Modelo'].nunique()} modelo(s) × {df['Pista'].nunique()} pistas")
    print(df.head(5).to_string(index=False))
    print("arquivos:", sorted(x.name for x in p.iterdir())[:8], "…")
    print()